<a href="https://colab.research.google.com/github/emmakelly7/ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [4]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
{'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
  ]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

One row of my result is one: line item from an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [5]:
import pandas as pd

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


I used pd.json_normalize() to flatten the nested lines data so that each row was one line item from an order. I set record_path='lines' because the items in the lines are the observations I want to be rows, and order and vendor_id were included with meta, so each line item stays with the correct original order and vendor. The output has four line items and a total quantity of 7. I used assertions to confirm the expected number of rows and the quantity total, which verifies that the data was flattened without losing any line items or quantities.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [6]:
vendors = pd.json_normalize(vendors_json)
joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


In [7]:
unmatched = joined[joined['_merge'] == 'left_only']
unmatched

,item,qty,order,vendor_id,name,zone,_merge
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


In [8]:
unmatched['qty'].sum()

np.int64(3)

I turned the vendor data into a dataframe. Then, I used vendor_id to merge it with the line item data that was flattened. I had to use a left join because I needed to keep all of the items from the orders even if the vendor was not recorded in the vendor table. I also used indicator=True, which made the _merge column so I could find  rows that did not match. Then, I filtered for left_only rows and used unmatched['qty'].sum() to find the sum of their quantities and output how many  were under an unmatched vendor. The output demonstrates that V-08 did not match a vendor that had been recorded, and it has 3 units.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [18]:
# The unmatched vendor will not appear because its name is NaN, and groupby() does not include missing group values.
vendor_totals = joined.groupby('name')['qty'].sum()
vendor_totals

,qty
name,
Cav Merch,1
Hoos Burgers,3


I used groupby('name')['qty'].sum() to find the total quantity for each matched vendor name. This output 3 units for Hoos Burgers and 1 unit for Cav Merch, which was a total quantity of 4 units. There is a total of 7 actual units, but the other 3 units are associated with the unmatched vendor V-08. V-08 did not have a corresponding record in the vendor table, so its name became NaN after the left join. groupby() does not show missing values, so V-08 and the 3 units do not appear in the totals grouped by vendor name.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [10]:
zone_totals = joined.groupby('zone')['qty'].sum()
zone_totals

,qty
zone,
A,4


The zone that sold the most items was Zone A with a quantity of 4. I used groupby('zone')['qty'].sum() to find the total number of items for  each zone. This total does not equal 7 because the 3 units from the unmatched vendor V-08 are not included. V-08 did not have a record in the vendor table, so the left join led to it being listed as (NaN) for zone. groupby() does not include missing group values, so those 3 units are not counted as part of the zone total. Therefore, the difference between the original 7 units and the 4 units assigned to Zone A is the 3 units associated with V-08.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [14]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

orders_with_lines = [
    order for order in ragged_json
    if 'lines' in order
]

ragged_lines = pd.json_normalize(
    orders_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


The ragged JSON contains different types of missing information, so I preserved those differences rather than treating them as equivalent. Order 5 does not contain a lines key, meaning there is no line-item record available to flatten, so I selected the orders containing lines before using pd.json_normalize(). Order 6 contains a line item for Fries but does not provide a qty, so the resulting quantity remains NaN. I did not replace this missing value with 0 because a missing quantity means the quantity is unknown or was not recorded, while 0 would represent a known quantity of zero. Preserving the missing value prevents the data from implying information that was not actually provided.

### Q6 — Validate

In [15]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a.) "One row per line item" is the useful shape here because each item and quantity can be evaluated on its own, and it can still stay in the correct order and the vendor's information. If there was only one row per order, multiple items would be nested together, which would make it more difficult to aggregate. If there is only one row per line item, you can use groupby to find things such as the total quantity sold by each vendor or zone more simply.

b.) In Q5, the practical difference between a missing quantity and quantity of zero is that a missing quantity means that the quantity was not recorded or it is unknown in the dataset. A quantity of zero is a recorded number that demonstrates that there were actually zero items of that type sold. It can turn into misleading information downstream if you conflate them. For example, if you treat a missing quantity as zero, you could be incorrectly assuming that there were zero items sold just because that information was not provided, which could be making an incorrect assumption. This could lead to some of the calculations being incorrect or misleading because the unknown quantities would be counted as zero rather than an actual number if they had one.